# 🚘 Distracted Driver Detection — Étape 1 : Préparation du Dataset

Ce notebook charge les images brutes du dataset State Farm, les prétraite (resize, normalisation, RGB) et les sauvegarde sous forme de tableaux NumPy prêts pour l'entraînement.

**Pipeline :**
1. Détection GPU/CPU
2. Chargement des images depuis `data/train/c0..c9/`
3. Resize 64×64 + normalisation [0, 1] + conversion RGB
4. Data Augmentation (optionnelle)
5. Sauvegarde `.npy` dans `data/processed/`
6. Visualisations (grille d'exemples + distribution des classes)

In [ ]:
# ─────────────────────────────────────────────
# IMPORTS
# ─────────────────────────────────────────────
import os
import time
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageEnhance
from pathlib import Path
from collections import Counter
from tqdm.notebook import tqdm

import tensorflow as tf

print(f"TensorFlow version : {tf.__version__}")
print(f"NumPy version      : {np.__version__}")

In [ ]:
# ─────────────────────────────────────────────
# DÉTECTION GPU / CPU
# ─────────────────────────────────────────────
gpus = tf.config.list_physical_devices('GPU')
cpus = tf.config.list_physical_devices('CPU')

print("=" * 50)
print(" RESSOURCES DISPONIBLES")
print("=" * 50)

if gpus:
    print(f"✅ GPU détecté(s) : {len(gpus)}")
    for gpu in gpus:
        print(f"   └── {gpu.name}")
    DEVICE = '/GPU:0'
    # Activer la croissance mémoire dynamique pour éviter les OOM
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print("   Croissance mémoire GPU activée.")
else:
    print(f"⚠️  Aucun GPU détecté. Utilisation CPU ({len(cpus)} unité(s)).")
    DEVICE = '/CPU:0'

print("=" * 50)

In [ ]:
# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────

# Chemin vers les images brutes (State Farm dataset)
DATA_PATH = Path("data/train")

# Dossier de sortie pour les tableaux NumPy prétraités
OUTPUT_PATH = Path("data/processed")
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

# Taille des images cible (doit correspondre à l'entrée du modèle CNN)
IMG_SIZE = (64, 64)

# ── Data Augmentation ──────────────────────────────────────────────────────────
# Mettre à True pour enrichir le dataset (recommandé si dataset déséquilibré)
ENABLE_AUGMENTATION = True
# Nombre de versions augmentées générées par image originale
AUGMENTATION_FACTOR = 2

# Graine aléatoire pour la reproductibilité
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Labels officiels State Farm
CLASS_NAMES = [
    "c0: Conduite normale",
    "c1: SMS au volant (Droit)",
    "c2: Téléphone (Droit)",
    "c3: SMS au volant (Gauche)",
    "c4: Téléphone (Gauche)",
    "c5: Réglage Radio",
    "c6: En train de Boire",
    "c7: Se retourner derrière",
    "c8: Maquillage",
    "c9: Parler au passager",
]
NUM_CLASSES = len(CLASS_NAMES)

print(f"Dataset source     : {DATA_PATH.resolve()}")
print(f"Sortie NumPy       : {OUTPUT_PATH.resolve()}")
print(f"Taille images      : {IMG_SIZE}")
print(f"Augmentation       : {'OUI (x' + str(AUGMENTATION_FACTOR) + ')' if ENABLE_AUGMENTATION else 'NON'}")
print(f"Nombre de classes  : {NUM_CLASSES}")

In [ ]:
# ─────────────────────────────────────────────
# FONCTIONS DE PRÉTRAITEMENT
# ─────────────────────────────────────────────

def load_and_preprocess(image_path: Path) -> np.ndarray:
    """
    Charge une image, la convertit en RGB, la redimensionne à IMG_SIZE,
    et normalise les valeurs de pixels dans [0, 1].
    Retourne un tableau NumPy de forme (64, 64, 3).
    """
    img = Image.open(image_path).convert("RGB")
    img = img.resize(IMG_SIZE, Image.BILINEAR)
    arr = np.array(img, dtype=np.float32) / 255.0
    return arr


def augment_image(arr: np.ndarray) -> list:
    """
    Applique des transformations aléatoires légères à un tableau image.
    Retourne une liste de tableaux augmentés.
    Transformations : flip horizontal, rotation légère, ajustement luminosité/contraste.
    """
    augmented = []
    img = Image.fromarray((arr * 255).astype(np.uint8))

    for _ in range(AUGMENTATION_FACTOR):
        aug = img.copy()

        # Flip horizontal (50% de chance)
        if random.random() > 0.5:
            aug = aug.transpose(Image.FLIP_LEFT_RIGHT)

        # Rotation légère (±15°)
        angle = random.uniform(-15, 15)
        aug = aug.rotate(angle, fillcolor=(0, 0, 0))

        # Ajustement luminosité (0.7 à 1.3)
        factor = random.uniform(0.7, 1.3)
        aug = ImageEnhance.Brightness(aug).enhance(factor)

        # Ajustement contraste (0.8 à 1.2)
        factor = random.uniform(0.8, 1.2)
        aug = ImageEnhance.Contrast(aug).enhance(factor)

        aug_arr = np.array(aug, dtype=np.float32) / 255.0
        augmented.append(aug_arr)

    return augmented


print("Fonctions de prétraitement définies.")

In [ ]:
# ─────────────────────────────────────────────
# CHARGEMENT ET PRÉTRAITEMENT
# ─────────────────────────────────────────────

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Le dossier '{DATA_PATH.resolve()}' est introuvable.\n"
        "Téléchargez le dataset State Farm depuis Kaggle et placez-le dans data/train/"
    )

SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}

X_list = []
y_list = []
class_counts = Counter()

# Collecte des chemins
all_items = []
for class_idx in range(NUM_CLASSES):
    class_dir = DATA_PATH / f"c{class_idx}"
    if not class_dir.exists():
        print(f"⚠️  Dossier manquant : {class_dir}")
        continue
    for img_path in class_dir.iterdir():
        if img_path.suffix.lower() in SUPPORTED_EXTENSIONS:
            all_items.append((img_path, class_idx))

print(f"Total d'images brutes trouvées : {len(all_items)}")
print("\nChargement et prétraitement en cours...")

start = time.time()
for img_path, label in tqdm(all_items, desc="Traitement des images"):
    try:
        arr = load_and_preprocess(img_path)
        X_list.append(arr)
        y_list.append(label)
        class_counts[label] += 1

        # Augmentation
        if ENABLE_AUGMENTATION:
            for aug_arr in augment_image(arr):
                X_list.append(aug_arr)
                y_list.append(label)
                class_counts[label] += 1

    except Exception as e:
        print(f"⚠️  Image ignorée ({img_path.name}) : {e}")

elapsed = time.time() - start
print(f"\n✅ Prétraitement terminé en {elapsed:.1f}s")
print(f"   Total images (avec augmentation) : {len(X_list)}")

In [ ]:
# ─────────────────────────────────────────────
# CONVERSION ET MÉLANGE
# ─────────────────────────────────────────────

X = np.array(X_list, dtype=np.float32)
y = np.array(y_list, dtype=np.int32)

# Mélange aléatoire reproductible
indices = np.arange(len(X))
np.random.shuffle(indices)
X = X[indices]
y = y[indices]

print(f"Shape X (images) : {X.shape}  — dtype : {X.dtype}")
print(f"Shape y (labels) : {y.shape}  — dtype : {y.dtype}")
print(f"Valeurs min/max pixels : {X.min():.3f} / {X.max():.3f}")

# Libération mémoire des listes
del X_list, y_list
print("\nListes intermédiaires libérées de la mémoire.")

In [ ]:
# ─────────────────────────────────────────────
# SAUVEGARDE .NPY
# ─────────────────────────────────────────────

x_path = OUTPUT_PATH / "X_processed.npy"
y_path = OUTPUT_PATH / "y_labels.npy"

print("Sauvegarde des tableaux NumPy...")
np.save(x_path, X)
np.save(y_path, y)

x_size_mb = x_path.stat().st_size / (1024 ** 2)
y_size_mb = y_path.stat().st_size / (1024 ** 2)

print(f"\n✅ Fichiers sauvegardés :")
print(f"   X_processed.npy → {x_size_mb:.1f} Mo — {X.shape}")
print(f"   y_labels.npy    → {y_size_mb:.2f} Mo — {y.shape}")
print(f"\n📂 Dossier de sortie : {OUTPUT_PATH.resolve()}")

In [ ]:
# ─────────────────────────────────────────────
# VISUALISATION : DISTRIBUTION DES CLASSES
# ─────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(14, 5))

counts = [class_counts.get(i, 0) for i in range(NUM_CLASSES)]
short_names = [n.split(":")[0] for n in CLASS_NAMES]
colors = plt.cm.viridis(np.linspace(0.2, 0.9, NUM_CLASSES))

bars = ax.bar(short_names, counts, color=colors, edgecolor='black', linewidth=0.5)
ax.set_title("Distribution des classes (après augmentation)", fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel("Classe")
ax.set_ylabel("Nombre d'images")
ax.set_facecolor('#f8f9fa')
fig.patch.set_facecolor('white')

for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width() / 2., bar.get_height() + 50,
            f'{count:,}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_PATH / "class_distribution.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Graphique sauvegardé dans {OUTPUT_PATH / 'class_distribution.png'}")

In [ ]:
# ─────────────────────────────────────────────
# VISUALISATION : GRILLE D'EXEMPLES PAR CLASSE
# ─────────────────────────────────────────────

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle("Exemples d'images prétraitées (64×64, normalisées)",
             fontsize=14, fontweight='bold', y=1.02)

for class_idx in range(NUM_CLASSES):
    ax = axes[class_idx // 5][class_idx % 5]
    
    # Trouver un exemple pour cette classe
    mask = (y == class_idx)
    if mask.any():
        sample_idx = np.where(mask)[0][0]
        ax.imshow(X[sample_idx])
        ax.set_title(CLASS_NAMES[class_idx].replace(": ", ":\n"), fontsize=8)
    else:
        ax.text(0.5, 0.5, "N/A", ha='center', va='center', transform=ax.transAxes)
    
    ax.axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_PATH / "sample_grid.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Grille sauvegardée dans {OUTPUT_PATH / 'sample_grid.png'}")

## ✅ Préparation Terminée

Les fichiers générés :

| Fichier | Description |
|---|---|
| `data/processed/X_processed.npy` | Tableau d'images normalisées, forme `(N, 64, 64, 3)` |
| `data/processed/y_labels.npy` | Labels correspondants, forme `(N,)`, valeurs 0–9 |
| `data/processed/class_distribution.png` | Histogramme de distribution des classes |
| `data/processed/sample_grid.png` | Grille visuelle d'exemples |

➡️ **Étape suivante** : Ouvrir le notebook `2-Training/2_Training.ipynb` pour entraîner le modèle CNN.